# Driven to Distraction: Extraneous Events and Underreaction to Earnings News

**Authors**: David Hirshleifer, Sonya S. Lim, Siew Hong Teoh

**Published**: 2009-09-28

**URL**: [https://doi.org/10.1111/j.1540-6261.2009.01501.x](https://doi.org/10.1111/j.1540-6261.2009.01501.x)

**Abstract**: Recent studies propose that limited investor attention causes market underreactions. This paper directly tests this explanation by measuring the information load faced by investors. The investor distraction hypothesis holds that extraneous news inhibits market reactions to relevant news. We find that the immediate price and volume reaction to a firm's earnings surprise is much weaker, and post‐announcement drift much stronger, when a greater number of same‐day earnings announcements are made by other firms. We evaluate the economic importance of distraction effects through a trading strategy, which yields substantial alphas. Industry‐unrelated news and large earnings surprises have a stronger distracting effect.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration parameters
symbol = 'AAPL'
start_date = '2010-01-01'
end_date = '2023-01-01'
lookback_period = 20


## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download historical data
data = yf.download(symbol, start=start_date, end=end_date)

# Feature engineering: Calculate daily returns
data['Return'] = data['Adj Close'].pct_change()

# Calculate moving average and standard deviation
data['MA'] = data['Adj Close'].rolling(window=lookback_period).mean()
data['STD'] = data['Adj Close'].rolling(window=lookback_period).std()


## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Generate trading signals based on moving average crossover
data['Signal'] = 0

data.loc[data['Adj Close'] > data['MA'], 'Signal'] = 1

data.loc[data['Adj Close'] < data['MA'], 'Signal'] = -1

# Portfolio construction: Calculate daily strategy returns
data['Strategy Return'] = data['Signal'].shift(1) * data['Return']


## Phase 4: Vectorized Backtest

In [ ]:
# Calculate cumulative returns
data['Cumulative Market Return'] = (1 + data['Return']).cumprod()
data['Cumulative Strategy Return'] = (1 + data['Strategy Return']).cumprod()


## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
sharpe_ratio = data['Strategy Return'].mean() / data['Strategy Return'].std() * np.sqrt(252)
sortino_ratio = data['Strategy Return'].mean() / data[data['Strategy Return'] < 0]['Strategy Return'].std() * np.sqrt(252)
max_drawdown = (data['Cumulative Strategy Return'].cummax() - data['Cumulative Strategy Return']).max()
calmar_ratio = data['Strategy Return'].mean() * 252 / max_drawdown

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Cumulative Market Return'], label='Market Return')
plt.plot(data['Cumulative Strategy Return'], label='Strategy Return')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Return')
plt.legend()
plt.show()

# Print performance metrics
print(f'Sharpe Ratio: {sharpe_ratio:.2f}')
print(f'Sortino Ratio: {sortino_ratio:.2f}')
print(f'Calmar Ratio: {calmar_ratio:.2f}')
print(f'Max Drawdown: {max_drawdown:.2%}')


## Phase 6: Monitoring Stub

In [ ]:
# Placeholder for monitoring code
# In practice, this would involve setting up alerts or dashboards to monitor strategy performance
pass
